In [2]:
import arcpy
import json
from collections import defaultdict

In [3]:
fc = r"D:\Documents\ArcGIS\Projects\ProyectoGrado\ProyectoGrado.gdb\lineas_vision_final_completo"

In [4]:
data = defaultdict(lambda: defaultdict(lambda: {
    "lenght": 0,
    "visible": 0
}))

In [5]:
fields = ["SOURCEOID", "OID_OBSERV", "VisCode", "TarIsVis", "Shape_Length", "OID_TARGET", "AZIMUTH", "VERT_ANGLE", "Shape_Length_1", "OBSTR_MPID"]

In [6]:
with arcpy.da.SearchCursor(fc, fields) as cursor: 
    for sourceoid, oid_observ, viscode, tarisvis, shape_length, oid_target, azimuth, vert_angle, shape_length_1, obstr_mpid in cursor: 
        data[oid_observ][oid_target]["lenght"] = shape_length_1 
        data[oid_observ][oid_target]["azimuth"] = azimuth 
        data[oid_observ][oid_target]["vert_angle"] = vert_angle 
        data[oid_observ][oid_target]["obstr_mpid"] = obstr_mpid 
        
        if tarisvis == 1: 
            data[oid_observ][oid_target]["visible"] = shape_length_1 
        elif viscode == 1: 
            data[oid_observ][oid_target]["visible"] += shape_length

In [7]:
#resultado = {}
camaras = []
max_length = 0
for oid_observer, sources in data.items():
    # resultado[oid_observer] = {
    #     "sources": {},
    #     "porcentaje_final": 0
    # }

    total_visibles = 0
    total_lineas = 0
    total_targets = 0
    lista_sources = []

    for sourceoid, vals in sources.items():
        total = vals["lenght"]
        visibles = vals["visible"]
        
        porcentaje = (visibles / total) * 100 if total > 0 else 0
        
        if porcentaje >= 90:
            lista_sources.append({
                "target_id": sourceoid,
                "azimuth": vals["azimuth"],
                "vert_angle": vals["vert_angle"],
                "obstr_mpid": vals["obstr_mpid"], 
                "length": total,
                "visible": visibles,
                "porcentaje": round(porcentaje, 2)
            })

        # resultado[oid_observer]["sources"][sourceoid] = {
        #     "azimuth": vals["azimuth"],
        #     "vert_angle": vals["vert_angle"],
        #     "length": total,
        #     "visible": visibles,
        #     "porcentaje": round(porcentaje, 2)
        # }
        if porcentaje >= 90:
            total_targets += 1

        total_visibles += visibles
        total_lineas += total
        
        


    # resultado[oid_observer]["total_targets"] = total_targets
    # resultado[oid_observer]["porcentaje_final"] = round(
    #     (total_visibles / total_lineas) * 100, 2
    # )
    if(total_lineas) > max_length:
        max_length = total_lineas
        
    camaras.append({
        "OID_OBSERVER": oid_observer,
        "porcentaje_final": round(
            (total_visibles / total_lineas) * 100, 2
        ),
        "total_length": total_lineas,
        "total_targets": total_targets,
        "sources": lista_sources
    })

In [8]:
for cam in camaras:
    cam["norm_length"] = cam["total_length"] / max_length 
    cam["score"] = 0.7 * cam["porcentaje_final"] + 0.3 * cam["norm_length"]

In [9]:
camaras_ordenadas = sorted(
    camaras,
    key=lambda x: (x["score"], x["total_targets"]),
    reverse=True
)

In [10]:
def seleccionar_camaras_minimas(camaras):
    # 1️⃣ Universo total de targets
    todos_los_targets = set()

    for camara in camaras:
        for src in camara["sources"]:
            todos_los_targets.add(src["target_id"])

    # 2️⃣ Selección greedy
    targets_cubiertos = set()
    camaras_seleccionadas = []

    for camara in camaras:
        oid = camara["OID_OBSERVER"]

        targets_camara = {
            src["target_id"]
            for src in camara["sources"]
        }

        nuevos_targets = targets_camara - targets_cubiertos

        # Solo se agrega si aporta cobertura nueva
        if nuevos_targets:
            camaras_seleccionadas.append({
                "OID_OBSERVER": oid,
                "porcentaje_final": camara["porcentaje_final"],
                "score": camara["score"],
                "targets_nuevos": list(nuevos_targets),
                "targets_totales": list(targets_camara)
            })

            targets_cubiertos.update(nuevos_targets)

        # 3️⃣ Condición de parada
        if targets_cubiertos == todos_los_targets:
            break

    return camaras_seleccionadas, targets_cubiertos, todos_los_targets

In [11]:
camaras_finales, cubiertos, total = seleccionar_camaras_minimas(camaras_ordenadas)

print(f"Cámaras necesarias: {len(camaras_finales)}")
print(f"Targets cubiertos: {len(cubiertos)} / {len(total)}")

Cámaras necesarias: 938
Targets cubiertos: 12813 / 12813


In [12]:
filtradas = []
for cam in camaras_finales:
    agregar = True
    for f in filtradas:
        # Calcular solapamiento con cámara ya agregada
        targets_comunes = set(cam["targets_totales"]) & set(f["targets_totales"])
        solapamiento = len(targets_comunes) / len(cam["targets_totales"])
        if solapamiento >= 0.70:
            agregar = False  # demasiada coincidencia, descartamos
            break
    if agregar:
        filtradas.append(cam)

In [13]:
oids = [item["OID_OBSERVER"] for item in filtradas]
oids_str = ",".join(map(str, oids))
sql = f"OID_OBSERVER IN ({oids_str})"
sql

'OID_OBSERVER IN (18599,17954,18780,18059,17769,18090,17933,18539,18928,18157,18881,18797,18391,17453,17730,18394,17645,18001,18690,17431,18296,17759,18813,18890,18407,18641,18398,18911,18297,17731,17638,18147,17817,18490,17863,17678,10433,18234,18174,18673,18889,14379,18711,18080,10772,18973,18024,2317,18892,604,18112,430,18374,18190,18922,17541,18743,17636,18904,8278,12168,18392,17982,17838,18355,13253,18781,18286,18384,18068,17832,18534,17809,17609,13540,2284,17551,17806,17937,5792,8480,18412,17442,18658,18920,6306,17985,17679,17528,18434,15165,17739,1944,17899,17790,18857,4407,17718,18153,18116,17426,15119,18091,17798,15114,18697,18513,17435,6130,4920,17920,3468,393,9171,17851,10976,17542,944,15120,18164,18314,18564,11345,7742,18259,17697,17795,950,18989,13569,15530,9714,17826,17630,14351,18332,15811,3668,13292,14640,17670,5505,11793,14937,15739,13165,17685,6706,5608,9265,17474,13612,17455,5605,11254,15805,14720,1208,18719,18200,4284,1596,14736,14309,18054,18556,7925,11859,4396,514

# GA

In [26]:
for cam in camaras:
    cam["targets_set"] = {
        src["target_id"]
        for src in cam["sources"]
        if src["visible"] >= 90
    }

In [46]:
def fitness(individuo, camaras, universo_targets):
    if not universo_targets:
        return -1e9

    targets_cubiertos = set()
    cams_activas = 0

    for cam in camaras:
        if individuo[cam["OID_OBSERVER"]] == 1:
            cams_activas += 1
            targets_cubiertos |= cam["targets_set"]

    cobertura = len(targets_cubiertos) / len(universo_targets)

    if cobertura == 0:
        return -1e9

    # ⚠ penalización FUERTE
    return (
        1000 * cobertura        # dominar la función
        - 5 * cams_activas      # castigar tamaño
    )

In [60]:
import random

def crear_individuo(camaras, max_cams=5000):
    individuo = {cam["OID_OBSERVER"]: 0 for cam in camaras}
    activos = random.sample(list(individuo.keys()), max_cams)
    for k in activos:
        individuo[k] = 1
    return individuo

def crear_poblacion(camaras, size=50):
    return [crear_individuo(camaras) for _ in range(size)]

In [58]:
def seleccion(poblacion, camaras, universo_targets, k=20):
    candidatos = random.sample(poblacion, k)
    return max(
        candidatos,
        key=lambda ind: fitness(ind, camaras, universo_targets)
    )

In [18]:
def crossover(p1, p2):
    hijo = {}
    for key in p1:
        hijo[key] = p1[key] if random.random() < 0.5 else p2[key]
    return hijo

In [56]:
#def mutar(individuo, tasa=0.02):
def mutar(individuo, tasa=0.9):
    for key in individuo:
        if random.random() < tasa:
            individuo[key] = 1 - individuo[key]

In [62]:
def algoritmo_genetico(camaras, generaciones=100, poblacion_size=50):
    universo_targets = set()
    for cam in camaras:
        universo_targets |= cam["targets_set"]
        
    print(len(universo_targets))

    poblacion = [crear_individuo(camaras) for _ in range(poblacion_size)]
    mejor = max(poblacion, key=lambda ind: fitness(ind, camaras, universo_targets))

    for gen in range(generaciones):
        nueva_poblacion = [mejor]  # 🔥 elitismo

        while len(nueva_poblacion) < poblacion_size:
            p1 = seleccion(poblacion, camaras, universo_targets)
            p2 = seleccion(poblacion, camaras, universo_targets)

            hijo = crossover(p1, p2)
            mutar(hijo, tasa=0.05)
            nueva_poblacion.append(hijo)

        poblacion = nueva_poblacion
        mejor = max(poblacion, key=lambda ind: fitness(ind, camaras, universo_targets))
        
        print(f"Fitness: {fitness(mejor, camaras, universo_targets):.2f} | ")
        
        if gen % 10 == 0:
            print(
                f"Gen {gen} | "
                f"Fitness: {fitness(mejor, camaras, universo_targets):.2f} | "
                f"Cámaras: {sum(mejor.values())}"
            )

    return mejor

In [63]:
mejor_solucion = algoritmo_genetico(camaras)

camaras_seleccionadas = [
    oid for oid, v in mejor_solucion.items() if v == 1
]

print("Cámaras seleccionadas:", camaras_seleccionadas)
print("Total:", len(camaras_seleccionadas))

10215


KeyboardInterrupt: 

In [64]:
def seleccionar_camaras_greedy(camaras, umbral=0.95):
    # 1️⃣ Crear el set de todos los targets
    todos_targets = set()
    for cam in camaras:
        todos_targets.update([s["target_id"] for s in cam["sources"]])
    
    # Objetivo: cubrir al menos umbral * total_targets
    total_a_cubrir = int(len(todos_targets) * umbral)
    
    # 2️⃣ Inicializar variables
    targets_cubiertos = set()
    camaras_seleccionadas = []
    
    # 3️⃣ Loop greedy
    while len(targets_cubiertos) < total_a_cubrir:
        mejor_camara = None
        targets_nuevos_max = 0
        
        for cam in camaras:
            # Targets que esta cámara puede cubrir y que aún no estén cubiertos
            targets_nuevos = set(s["target_id"] for s in cam["sources"]) - targets_cubiertos
            if len(targets_nuevos) > targets_nuevos_max:
                targets_nuevos_max = len(targets_nuevos)
                mejor_camara = cam
        
        if not mejor_camara:
            # Ya no quedan cámaras que agreguen nuevos targets
            break
        
        # Añadir cámara seleccionada
        camaras_seleccionadas.append(mejor_camara)
        # Marcar targets cubiertos
        targets_cubiertos.update(s["target_id"] for s in mejor_camara["sources"])
        # Opcional: remover la cámara de la lista para no reconsiderarla
        camaras.remove(mejor_camara)
        
        print(f"Seleccionada cámara {mejor_camara['OID_OBSERVER']}, "
              f"cubriendo {targets_nuevos_max} targets nuevos, "
              f"{len(targets_cubiertos)}/{total_a_cubrir} cubiertos")
    
    print(f"\nTotal cámaras seleccionadas: {len(camaras_seleccionadas)}")
    print(f"Porcentaje de targets cubiertos: {len(targets_cubiertos)/len(todos_targets)*100:.2f}%")
    
    return camaras_seleccionadas

In [65]:
seleccionadas = seleccionar_camaras_greedy(camaras, umbral=0.95)

Seleccionada cámara 14329, cubriendo 584 targets nuevos, 584/12172 cubiertos
Seleccionada cámara 18265, cubriendo 490 targets nuevos, 1074/12172 cubiertos
Seleccionada cámara 17678, cubriendo 374 targets nuevos, 1448/12172 cubiertos
Seleccionada cámara 17952, cubriendo 351 targets nuevos, 1799/12172 cubiertos
Seleccionada cámara 6303, cubriendo 341 targets nuevos, 2140/12172 cubiertos
Seleccionada cámara 17415, cubriendo 313 targets nuevos, 2453/12172 cubiertos
Seleccionada cámara 17382, cubriendo 284 targets nuevos, 2737/12172 cubiertos
Seleccionada cámara 10433, cubriendo 279 targets nuevos, 3016/12172 cubiertos
Seleccionada cámara 18959, cubriendo 273 targets nuevos, 3289/12172 cubiertos
Seleccionada cámara 4842, cubriendo 256 targets nuevos, 3545/12172 cubiertos
Seleccionada cámara 12168, cubriendo 252 targets nuevos, 3797/12172 cubiertos
Seleccionada cámara 17900, cubriendo 237 targets nuevos, 4034/12172 cubiertos
Seleccionada cámara 17435, cubriendo 230 targets nuevos, 4264/12172

In [67]:
oids = [item["OID_OBSERVER"] for item in seleccionadas]
oids_str = ",".join(map(str, oids))
sql = f"OID_OBSERVER IN ({oids_str})"
sql

'OID_OBSERVER IN (14329,18265,17678,17952,6303,17415,17382,10433,18959,4842,12168,17900,17435,17670,1208,5792,17300,7790,15495,2354,2731,15812,2284,17508,18883,18164,18916,17521,3668,18286,4457,17723,17795,1914,12822,17442,12867,14735,18077,18157,928,6811,18227,5165,15181,17892,1008,12912,8867,9714,4873,18653,3939,16310,3753,12924,18623,4059,7047,5608,6822,839,3690,13393,17977,17568,14779,9016,1096,18611,4584,16570,11919,5556,14016,14378,7743,16435,10772,11723,18134,18556,7094,5658,7235,5946,8042,15805,18543,7535,12324,16029,1694,6728,8606,13253,16999,9184,9435,18355,10434,18102,2811,7497,9209,700,18772,3531,5951,7063,11435,13903,17789,7155,12183,14634,4838,5102,3268,9286,10349,18087,18588,272,474,718,18296,666,3815,8261,12092,14306,16302,2482)'